# UPI Transaction Fraud Detection

## 1. Business Context
India's Unified Payments Interface (UPI) processes an average of **66 crore transactions daily** (totaling ₹314 lakh crore annually), making it the backbone of digital payments for Indian consumers and merchants. As adoption grows, payment platforms face rising pressure to protect users without slowing down everyday transactions.

## 2. Problem Statement
Digital payment fraud is a growing threat in India, with over **10.64 lakh UPI fraud cases worth ₹805 crore** reported in a single fiscal year. Fraudsters use stolen credentials and social engineering to execute unauthorized transfers. 

For a mid-sized Indian payment platform losing an estimated **₹12 lakh per month** to undetected scams, the challenge is two-fold:
- **Missed Fraud (False Negatives):** Direct financial loss and loss of merchant/customer trust.
- **False Alarms (False Positives):** Blocking genuine ₹300–₹500 daily payments frustrates users and drives them to competitor apps.

## 3. Project Objective
Build a machine learning system to score every UPI transaction in real time and categorize it as **Genuine** or **Fraud**. High-risk transactions will be flagged for step-up verification (such as an extra OTP or biometric check) before the transfer is completed.

## 4. ML Problem Formulation
- **Task Type:** Supervised Binary Classification (`0` = Genuine, `1` = Fraud)
- **Dataset:** UPI Transactions 2024 (Kaggle) — 2,50,000 rows with extreme class imbalance (~0.2% fraud rate).

## 5. Success Metrics
- **Primary ML Metrics:** **Recall** (catch at least 85% of actual fraud cases) and **PR-AUC** (appropriate for imbalanced data).
- **Business Metric:** Reduce monthly fraud losses by at least **₹8 lakh** while keeping false alarms under **2%** of genuine transactions.

## 6. Key Constraints
- **Low Latency:** Predictions must run in milliseconds to support real-time UPI processing.
- **Explainability:** The fraud operations team requires clear, human-readable reasons (using SHAP) for why a transaction was flagged.

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
RANDOM_STATE=42
pd.set_option('display.max_columns',None)

In [14]:
data=pd.read_csv("/home/saurabh/AI_PROJECTS/UPI_fraud_prediction/upi_transactions_2024.csv")

In [15]:
data.head()

,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend
0,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0
1,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0
2,TXN0000000003,2024-04-02 13:27:18,P2P,Grocery,477,SUCCESS,26-35,36-45,Karnataka,Yes Bank,PNB,Android,4G,0,13,Tuesday,0
3,TXN0000000004,2024-01-07 10:09:17,P2P,Fuel,2784,SUCCESS,26-35,26-35,Delhi,ICICI,PNB,Android,5G,0,10,Sunday,1
4,TXN0000000005,2024-01-23 19:04:23,P2P,Shopping,990,SUCCESS,26-35,18-25,Delhi,Axis,Yes Bank,iOS,WiFi,0,19,Tuesday,0


In [16]:
data

,transaction id,timestamp,transaction type,merchant_category,amount (INR),transaction_status,sender_age_group,receiver_age_group,sender_state,sender_bank,receiver_bank,device_type,network_type,fraud_flag,hour_of_day,day_of_week,is_weekend
0,TXN0000000001,2024-10-08 15:17:28,P2P,Entertainment,868,SUCCESS,26-35,18-25,Delhi,Axis,SBI,Android,4G,0,15,Tuesday,0
1,TXN0000000002,2024-04-11 06:56:00,P2M,Grocery,1011,SUCCESS,26-35,26-35,Uttar Pradesh,ICICI,Axis,iOS,4G,0,6,Thursday,0
2,TXN0000000003,2024-04-02 13:27:18,P2P,Grocery,477,SUCCESS,26-35,36-45,Karnataka,Yes Bank,PNB,Android,4G,0,13,Tuesday,0
3,TXN0000000004,2024-01-07 10:09:17,P2P,Fuel,2784,SUCCESS,26-35,26-35,Delhi,ICICI,PNB,Android,5G,0,10,Sunday,1
4,TXN0000000005,2024-01-23 19:04:23,P2P,Shopping,990,SUCCESS,26-35,18-25,Delhi,Axis,Yes Bank,iOS,WiFi,0,19,Tuesday,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
249995,TXN0000249996,2024-11-08 22:41:43,Recharge,Food,373,SUCCESS,36-45,26-35,Telangana,ICICI,HDFC,iOS,5G,0,22,Friday,0
249996,TXN0000249997,2024-12-15 02:58:03,P2P,Utilities,2025,SUCCESS,36-45,26-35,Rajasthan,HDFC,Yes Bank,Android,5G,0,2,Sunday,1
249997,TXN0000249998,2024-11-27 16:33:25,P2P,Food,468,SUCCESS,26-35,18-25,West Bengal,ICICI,ICICI,Android,4G,0,16,Wednesday,0
249998,TXN0000249999,2024-01-05 13:31:30,Recharge,Healthcare,284,SUCCESS,18-25,36-45,Andhra Pradesh,Axis,ICICI,iOS,4G,0,13,Friday,0


In [18]:
data.shape

(250000, 17)

In [19]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 17 columns):
 #   Column              Non-Null Count   Dtype 
---  ------              --------------   ----- 
 0   transaction id      250000 non-null  object
 1   timestamp           250000 non-null  object
 2   transaction type    250000 non-null  object
 3   merchant_category   250000 non-null  object
 4   amount (INR)        250000 non-null  int64 
 5   transaction_status  250000 non-null  object
 6   sender_age_group    250000 non-null  object
 7   receiver_age_group  250000 non-null  object
 8   sender_state        250000 non-null  object
 9   sender_bank         250000 non-null  object
 10  receiver_bank       250000 non-null  object
 11  device_type         250000 non-null  object
 12  network_type        250000 non-null  object
 13  fraud_flag          250000 non-null  int64 
 14  hour_of_day         250000 non-null  int64 
 15  day_of_week         250000 non-null  object
 16  is

In [21]:
data.isnull().sum()

transaction id        0
timestamp             0
transaction type      0
merchant_category     0
amount (INR)          0
transaction_status    0
sender_age_group      0
receiver_age_group    0
sender_state          0
sender_bank           0
receiver_bank         0
device_type           0
network_type          0
fraud_flag            0
hour_of_day           0
day_of_week           0
is_weekend            0
dtype: int64

In [23]:
data['merchant_category'].value_counts()

merchant_category
Grocery          49966
Food             37464
Shopping         29872
Fuel             25063
Other            24828
Utilities        22338
Transport        20105
Entertainment    20103
Healthcare       12663
Education         7598
Name: count, dtype: int64

In [24]:
data['transaction_status'].value_counts()

transaction_status
SUCCESS    237624
FAILED      12376
Name: count, dtype: int64

In [25]:
data['sender_age_group'].value_counts()

sender_age_group
26-35    87432
36-45    62873
18-25    62345
46-55    24841
56+      12509
Name: count, dtype: int64

In [28]:
data['receiver_age_group'].value_counts()

receiver_age_group
26-35    87864
18-25    62611
36-45    62151
46-55    24823
56+      12551
Name: count, dtype: int64

In [31]:
data['sender_state'].value_counts()

sender_state
Maharashtra       37427
Uttar Pradesh     30125
Karnataka         29756
Tamil Nadu        25367
Delhi             24870
Telangana         22435
Gujarat           20061
Andhra Pradesh    20006
Rajasthan         19981
West Bengal       19972
Name: count, dtype: int64

In [32]:
data['sender_bank'].value_counts()

sender_bank
SBI         62693
HDFC        37485
ICICI       29769
IndusInd    25173
Axis        25042
PNB         24946
Yes Bank    24860
Kotak       20032
Name: count, dtype: int64

In [33]:
data['device_type'].value_counts()

device_type
Android    187777
iOS         49613
Web         12610
Name: count, dtype: int64

In [34]:
data['network_type'].value_counts()

network_type
4G      149813
5G       62582
WiFi     25134
3G       12471
Name: count, dtype: int64

In [35]:
data['fraud_flag'].value_counts(normalize=True)*100

fraud_flag
0    99.808
1     0.192
Name: proportion, dtype: float64